In [ ]:
# ============================================================
# CELL 1: IMPORT REQUIRED LIBRARIES
# ============================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

In [ ]:
# ============================================================
# CELL 2: UPLOAD AND LOAD DATASET
# ============================================================

from google.colab import files
import zipfile
import os

# Upload the dataset ZIP file
uploaded = files.upload()

# Get the uploaded ZIP filename
zip_file = list(uploaded.keys())[0]

print("Uploaded file:", zip_file)

# Extract the ZIP file
extract_folder = "/content/customer_transaction_data"

from pathlib import Path

extract_folder = Path("/content/customer_transaction_data").resolve()

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    for member in zip_ref.infolist():
        target_path = (extract_folder / member.filename).resolve()
        if target_path != extract_folder and extract_folder not in target_path.parents:
            raise ValueError("Unsafe file path found in ZIP archive")
        zip_ref.extract(member, extract_folder)

print("Dataset extracted successfully!")

# Display extracted files
for root, dirs, files_list in os.walk(extract_folder):
    for file in files_list:
        print(os.path.join(root, file))

In [ ]:

# ============================================================
# CELL 3: LOAD AND INSPECT THE DATASET
# ============================================================

import pandas as pd
import os

# Define the dataset path
file_path = "/content/customer_transaction_data/Data/train(1).csv"

# Check whether the file exists
if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"Dataset not found at: {file_path}"
    )

# Load the CSV file
df = pd.read_csv(file_path)

# Display the first five rows
print("First 5 rows of the dataset:")
display(df.head())

# Display dataset dimensions
print("\nDataset shape:")
print(df.shape)

# Display column names
print("\nColumn names:")
print(df.columns.tolist())

# Display data types and non-null counts
print("\nDataset information:")
df.info()


In [ ]:

# ============================================================
# CELL 4: DATA QUALITY CHECKS
# ============================================================

# 1. Check missing values
print("1. Missing values:")
total_missing = df.isnull().sum().sum()
print("Total missing values:", total_missing)

# Display columns with missing values, if any
missing_by_column = df.isnull().sum()
print("\nColumns containing missing values:")
display(missing_by_column[missing_by_column > 0])

# 2. Check duplicate rows
print("\n2. Duplicate rows:")
duplicate_count = df.duplicated().sum()
print("Total duplicate rows:", duplicate_count)

# 3. Check duplicate customer IDs
print("\n3. Duplicate customer IDs:")
duplicate_id_count = df["ID_code"].duplicated().sum()
print("Total duplicate IDs:", duplicate_id_count)

# 4. Check target distribution
print("\n4. Target distribution:")
display(df["target"].value_counts().sort_index())

# 5. Check target percentages
print("\n5. Target percentages:")
target_percentages = (
    df["target"].value_counts(normalize=True).sort_index() * 100
)
display(target_percentages.round(2))

# 6. Verify target values
print("\n6. Unique target values:")
print(sorted(df["target"].dropna().unique()))


In [ ]:

# ============================================================
# CELL 5: SEPARATE FEATURES AND TARGET
# ============================================================

# Separate the target column
y = df["target"].astype(int)

# Remove ID_code and target from the input features
X = df.drop(columns=["ID_code", "target"])

# Display the shapes
print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

# Verify feature columns
print("\nFirst 10 feature columns:")
print(X.columns[:10].tolist())

# Verify target values
print("\nTarget classes:")
print(y.value_counts().sort_index())

# Confirm that IDs and target are not in X
assert "ID_code" not in X.columns
assert "target" not in X.columns

print("\nFeature and target separation completed successfully!")


In [ ]:

# ============================================================
# CELL 6: TRAIN-TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

# Split the data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Display the dimensions of each dataset
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target:", y_train.shape)
print("Testing target:", y_test.shape)

# Check target distribution in training data
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).sort_index().mul(100).round(2))

# Check target distribution in testing data
print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).sort_index().mul(100).round(2))

print("\nTrain-test split completed successfully!")


In [ ]:

# ============================================================
# CELL 7: LOGISTIC REGRESSION BASELINE MODEL
# ============================================================

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Create a pipeline:
# 1. Standardize the features
# 2. Train Logistic Regression
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
)

# Train the model using training data only
logistic_model.fit(X_train, y_train)

# Predict class labels for the test set
y_pred_lr = logistic_model.predict(X_test)

# Predict probabilities for class 1
y_prob_lr = logistic_model.predict_proba(X_test)[:, 1]

# Evaluate the model
print("LOGISTIC REGRESSION RESULTS")
print("-" * 40)

print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_lr, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_lr, zero_division=0):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, zero_division=0))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))


In [ ]:

# ============================================================
# CELL 8: DECISION TREE CLASSIFIER
# ============================================================

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Create the Decision Tree model
decision_tree_model = DecisionTreeClassifier(
    max_depth=12,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42
)

# Train the model
decision_tree_model.fit(X_train, y_train)

# Predict class labels
y_pred_dt = decision_tree_model.predict(X_test)

# Predict probabilities for class 1
y_prob_dt = decision_tree_model.predict_proba(X_test)[:, 1]

# Evaluate the model
print("DECISION TREE RESULTS")
print("-" * 40)

print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_dt, zero_division=0):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_dt):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt, zero_division=0))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))


In [ ]:

# ============================================================
# CELL 9: RANDOM FOREST CLASSIFIER
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Create the Random Forest model
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=10,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

# Train the model
random_forest_model.fit(X_train, y_train)

# Predict class labels
y_pred_rf = random_forest_model.predict(X_test)

# Predict probabilities for class 1
y_prob_rf = random_forest_model.predict_proba(X_test)[:, 1]

# Evaluate the model
print("RANDOM FOREST RESULTS")
print("-" * 40)

print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_rf, zero_division=0):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_rf):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))


In [ ]:

# ============================================================
# CELL 10: HISTOGRAM GRADIENT BOOSTING
# ============================================================

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Calculate balanced sample weights from training labels only
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

# Create the model
gradient_boosting_model = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.1,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

# Train using balanced sample weights
gradient_boosting_model.fit(
    X_train,
    y_train,
    sample_weight=sample_weights
)

# Predict class labels
y_pred_gb = gradient_boosting_model.predict(X_test)

# Predict probabilities for class 1
y_prob_gb = gradient_boosting_model.predict_proba(X_test)[:, 1]

# Evaluate the model
print("HISTOGRAM GRADIENT BOOSTING RESULTS")
print("-" * 45)

print(f"Accuracy:  {accuracy_score(y_test, y_pred_gb):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_gb, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_gb, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred_gb, zero_division=0):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_gb):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_gb, zero_division=0))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_gb))


In [ ]:

# ============================================================
# CELL 11: MODEL COMPARISON REPORT
# ============================================================

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Collect predictions from all four models
model_results = {
    "Logistic Regression": (y_pred_lr, y_prob_lr),
    "Decision Tree": (y_pred_dt, y_prob_dt),
    "Random Forest": (y_pred_rf, y_prob_rf),
    "HistGradientBoosting": (y_pred_gb, y_prob_gb)
}

# Calculate evaluation metrics
comparison = []

for model_name, (predictions, probabilities) in model_results.items():
    comparison.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1-Score": f1_score(
            y_test, predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

# Create and display the comparison table
comparison_df = pd.DataFrame(comparison)

comparison_df = comparison_df.sort_values(
    by="F1-Score",
    ascending=False
).reset_index(drop=True)

print("MODEL COMPARISON REPORT")
display(comparison_df.round(4))

# Identify the best model by F1-score
best_model_name = comparison_df.iloc[0]["Model"]

print("\nBest model by F1-score:", best_model_name)

# Save the report as a CSV file
comparison_df.to_csv(
    "/content/model_comparison_report.csv",
    index=False
)

print("\nComparison report saved successfully.")


In [ ]:

# ============================================================
# CELL 12: CREATE TRAINING AND VALIDATION SETS
# ============================================================

from sklearn.model_selection import train_test_split

# Split the existing training data
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("Model training features:", X_train_sub.shape)
print("Validation features:", X_val.shape)

print("\nModel training target distribution:")
print(y_train_sub.value_counts(normalize=True).sort_index().round(4))

print("\nValidation target distribution:")
print(y_val.value_counts(normalize=True).sort_index().round(4))

print("\nValidation split completed successfully!")


In [ ]:

from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.ensemble import HistGradientBoostingClassifier
import pandas as pd

# Give more weight to the minority class (target = 1)
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_sub
)

# Try four different model configurations
parameter_options = [
    {"max_iter": 200, "learning_rate": 0.1, "max_leaf_nodes": 31, "l2_regularization": 1.0},
    {"max_iter": 300, "learning_rate": 0.05, "max_leaf_nodes": 31, "l2_regularization": 1.0},
    {"max_iter": 200, "learning_rate": 0.1, "max_leaf_nodes": 15, "l2_regularization": 5.0},
    {"max_iter": 300, "learning_rate": 0.1, "max_leaf_nodes": 31, "l2_regularization": 5.0}
]

tuning_results = []
trained_candidates = []

for i, params in enumerate(parameter_options, start=1):
    print(f"Training configuration {i}/4: {params}")

    model = HistGradientBoostingClassifier(
        **params,
        random_state=42
    )

    model.fit(
        X_train_sub,
        y_train_sub,
        sample_weight=sample_weights
    )

    val_predictions = model.predict(X_val)
    val_probabilities = model.predict_proba(X_val)[:, 1]

    tuning_results.append({
        "Configuration": i,
        "Accuracy": accuracy_score(y_val, val_predictions),
        "Precision": precision_score(
            y_val, val_predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, val_predictions, zero_division=0
        ),
        "F1-Score": f1_score(
            y_val, val_predictions, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_val, val_probabilities),
        "Parameters": params
    })

    trained_candidates.append(model)

# Rank configurations by validation F1-score
tuning_df = pd.DataFrame(tuning_results)
tuning_df = tuning_df.sort_values(
    by="F1-Score",
    ascending=False
).reset_index(drop=True)

display(tuning_df.round(4))

best_configuration = int(tuning_df.iloc[0]["Configuration"])
best_params = parameter_options[best_configuration - 1]
best_validation_model = trained_candidates[best_configuration - 1]

print("Best configuration:", best_configuration)
print("Best parameters:", best_params)
print("Best validation F1-score:", round(tuning_df.iloc[0]["F1-Score"], 4))


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

# Get positive-class probabilities on the validation set
val_probabilities = best_validation_model.predict_proba(X_val)[:, 1]

# Test different classification thresholds
thresholds = np.arange(0.20, 0.71, 0.05)
threshold_results = []

for threshold in thresholds:
    val_predictions = (val_probabilities >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(float(threshold), 2),
        "Precision": precision_score(
            y_val, val_predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, val_predictions, zero_division=0
        ),
        "F1-Score": f1_score(
            y_val, val_predictions, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df = threshold_df.sort_values(
    by="F1-Score",
    ascending=False
).reset_index(drop=True)

display(threshold_df.round(4))

# Select the threshold with the highest validation F1-score
best_threshold = float(threshold_df.iloc[0]["Threshold"])

print("Best threshold:", best_threshold)
print("Best threshold validation F1-score:",
      round(float(threshold_df.iloc[0]["F1-Score"]), 4))


In [ ]:

from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from sklearn.ensemble import HistGradientBoostingClassifier

# Calculate sample weights using the complete training split
final_sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

# Create the final model with the selected hyperparameters
final_model = HistGradientBoostingClassifier(
    **best_params,
    random_state=42
)

# Train on all 160,000 training rows
final_model.fit(
    X_train,
    y_train,
    sample_weight=final_sample_weights
)

# Predict probabilities on the original test split
y_test_probabilities = final_model.predict_proba(X_test)[:, 1]

# Apply the threshold selected using the validation set
y_test_predictions = (
    y_test_probabilities >= best_threshold
).astype(int)

# Evaluate the model
print("Final model evaluation")
print("----------------------")
print("Threshold:", best_threshold)
print("Accuracy:", round(accuracy_score(y_test, y_test_predictions), 4))
print("Precision:", round(precision_score(y_test, y_test_predictions, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_test_predictions, zero_division=0), 4))
print("F1-score:", round(f1_score(y_test, y_test_predictions, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_test_probabilities), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_predictions))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_predictions,
    target_names=["No Transaction (0)", "Transaction (1)"],
    zero_division=0
))


In [ ]:

import joblib
import os

# Bundle everything required for future predictions
model_bundle = {
    "model": final_model,
    "feature_columns": list(X.columns),
    "threshold": best_threshold,
    "model_name": "HistGradientBoostingClassifier"
}

# Save the model bundle
model_path = "/content/customer_transaction_model.joblib"

joblib.dump(model_bundle, model_path)

# Verify that the saved file can be loaded
loaded_bundle = joblib.load(model_path)

print("Model saved successfully!")
print("File path:", model_path)
print("File size:", round(os.path.getsize(model_path) / (1024 * 1024), 2), "MB")
print("Model:", loaded_bundle["model_name"])
print("Number of features:", len(loaded_bundle["feature_columns"]))
print("Classification threshold:", loaded_bundle["threshold"])


In [ ]:

# Save the model comparison table
comparison_report_path = "/content/model_comparison_report.csv"

comparison_df.to_csv(comparison_report_path, index=False)

print("Model comparison report saved!")
print("File path:", comparison_report_path)

display(comparison_df.round(4))


In [ ]:

challenges_report = """
CUSTOMER TRANSACTION PREDICTION
PROJECT CHALLENGES AND OBSERVATIONS

1. Class Imbalance
The target distribution contains approximately 89.95% class 0
and 10.05% class 1. This imbalance can make accuracy misleading.
Class weighting and sample weighting were used for selected models.

2. Anonymized Features
The dataset contains 200 anonymized numeric features. Their business
meanings are not available, limiting domain-specific interpretation.

3. Model Comparison
Logistic Regression, Decision Tree, Random Forest, and
HistGradientBoosting were compared. HistGradientBoosting achieved
the strongest F1-score among the original model configurations.

4. Random Forest Limitation
The original Random Forest configuration achieved 90.10% accuracy,
but only 1.74% recall for class 1. This shows why accuracy alone
is not sufficient for imbalanced classification.

5. Threshold Selection
The default threshold of 0.50 was compared with alternative
thresholds on the validation set. A threshold of 0.65 produced
the highest validation F1-score among the thresholds tested.

6. Final Model Results
Final test-split metrics at threshold 0.65:
Accuracy: 0.9053
Precision: 0.5276
Recall: 0.5507
F1-score: 0.5389
ROC-AUC: 0.8858

7. Evaluation Limitation
The original test split was examined during earlier model comparisons.
Therefore, it should not be described as a completely untouched final
evaluation. A fresh holdout or properly designed cross-validation
would provide a stronger unbiased estimate of generalization.

8. Future Improvements
- Test additional hyperparameters using validation data.
- Consider precision-recall curves and business-specific thresholds.
- Evaluate calibration and operational costs of false positives
  and false negatives.
- Validate the model on a fresh, untouched dataset.
- Monitor performance after deployment.

CONCLUSION
HistGradientBoosting was selected based on F1-score, with a
validation-selected threshold of 0.65. The model achieved an
F1-score of 0.5389 and ROC-AUC of 0.8858 on the existing test split.
Further validation and threshold assessment may improve its practical
usefulness.
"""

challenges_path = "/content/project_challenges_report.txt"

with open(challenges_path, "w", encoding="utf-8") as file:
    file.write(challenges_report.strip())

print("Challenges report saved!")
print("File path:", challenges_path)
